Goal: detect face/pose/hands; crop tight ROI; write to videos_roi/; build manifest_nslt2000_roi.csv.

In [ ]:
# Cell A — Setup MediaPipe ROI crop
from pathlib import Path
import numpy as np, cv2, pandas as pd

root = Path("..").resolve()
data_dir = root/"data"/"wlasl_preprocessed"
man_src = data_dir/"manifest_nslt2000_trim.csv"  # use trimmed sources
df = pd.read_csv(man_src)

import mediapipe as mp
mp_pose = mp.solutions.pose
mp_hands = mp.solutions.hands
mp_face  = mp.solutions.face_detection

def detect_roi_bounding_box(frames_bgr):
    """Return (x1,y1,x2,y2) covering upper-body + hands across frames."""
    H, W = frames_bgr[0].shape[:2]
    xs, ys, xe, ye = [], [], [], []
    with mp_pose.Pose(static_image_mode=False, model_complexity=0) as pose, \
         mp_hands.Hands(static_image_mode=False, max_num_hands=2) as hands, \
         mp_face.FaceDetection(model_selection=0) as face:
        step = max(1, len(frames_bgr)//12)
        for f in frames_bgr[::step]:
            rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            pr = pose.process(rgb)
            hr = hands.process(rgb)
            fr = face.process(rgb)
            def norm_to_px(lm):
                return int(lm.x*W), int(lm.y*H)
            # Pose upper body landmarks
            if pr.pose_landmarks:
                for lm in pr.pose_landmarks.landmark:
                    x,y = norm_to_px(lm)
                    xs+= [x]; ys+= [y]; xe+= [x]; ye+= [y]
            # Hands
            if hr.multi_hand_landmarks:
                for h in hr.multi_hand_landmarks:
                    for lm in h.landmark:
                        x,y = norm_to_px(lm); xs+= [x]; ys+= [y]; xe+= [x]; ye+= [y]
            # Face box
            if fr.detections:
                for d in fr.detections:
                    bb = d.location_data.relative_bounding_box
                    x1,y1 = int(bb.xmin*W), int(bb.ymin*H)
                    x2,y2 = int((bb.xmin+bb.width)*W), int((bb.ymin+bb.height)*H)
                    xs+= [x1]; ys+= [y1]; xe+= [x2]; ye+= [y2]
    if not xs:
        # fallback to center upper body region
        x1,y1,x2,y2 = int(W*0.2), int(H*0.2), int(W*0.8), int(H*0.9)
    else:
        x1,y1,x2,y2 = max(0,min(xs)), max(0,min(ys)), min(W,max(xe)), min(H,max(ye))
        # pad a bit
        padx, pady = int(0.05*W), int(0.05*H)
        x1,y1,x2,y2 = max(0,x1-padx), max(0,y1-pady), min(W,x2+padx), min(H,y2+pady)
    return x1,y1,x2,y2

def read_video_cv(path):
    cap = cv2.VideoCapture(str(path))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok: break
        frames.append(frame)  # BGR
    cap.release()
    return frames

vroi = data_dir/"videos_roi"
vroi.mkdir(exist_ok=True)


In [ ]:
# Cell B — Crop and write ROI videos; build manifest_nslt2000_roi.csv
import shlex, subprocess

def crop_and_write(src, dst):
    frames = read_video_cv(src)
    if not frames: return False
    x1,y1,x2,y2 = detect_roi_bounding_box(frames)
    H, W = frames[0].shape[:2]
    # Use ffmpeg crop for speed/quality
    crop = f"crop={x2-x1}:{y2-y1}:{x1}:{y1},scale=-1:256,fps=30"
    cmd = f"ffmpeg -y -loglevel error -i {shlex.quote(str(src))} -vf {crop} -an -c:v libx264 -pix_fmt yuv420p -preset veryfast -crf 23 -movflags +faststart {shlex.quote(str(dst))}"
    subprocess.run(cmd, shell=True, check=True)

for p_rel in df["path"].unique():
    src = root/p_rel
    dst = vroi/src.name
    if dst.exists(): continue
    try:
        crop_and_write(src, dst)
    except Exception as e:
        print("Crop failed:", src.name, e)

man_roi = data_dir/"manifest_nslt2000_roi.csv"
df_roi = df.copy()
def prefer_roi(p_rel):
    dst = data_dir/"videos_roi"/Path(p_rel).name
    return dst.relative_to(root).as_posix() if dst.exists() else p_rel
df_roi["path"] = df_roi["path"].map(prefer_roi)
df_roi.to_csv(man_roi, index=False)
print("Saved:", man_roi)
